### Sequential Agent in Andromeda

In [48]:
from typing import TypedDict,Optional
from pydantic import BaseModel
from andromeda.core.agent import Agent
from andromeda.core.workflow import WorkflowBuilder
from andromeda.config import AgentConfig, ModelConfig
from andromeda import HumanMessage
from pprint import pprint

In [31]:
modelconfig = ModelConfig(name = 'llama3.2:3b', provider = 'ollama', temperature = 0.5)

In [32]:
class AgentState(BaseModel):
    query: str
    researcher: Optional[str] = None
    outliner: Optional[str] = None
    writer: Optional[str] = None
    review : Optional[str] = None

In [33]:
researcher = Agent(
    AgentConfig(
        name = "researcher",
        model = modelconfig,
        prompt = "You research a topic and list 3-5 facts. Be concise and to the point. STOP once done"
    )
)

In [34]:
outliner = Agent(
    AgentConfig(
        name = "outliner",
        model = modelconfig,
        prompt = "You turn research notes into short sturectured outline (Heading only). STOP once done"
    )
)

In [35]:
writer = Agent(
    AgentConfig(
        name = "writer",
        model = modelconfig,
        prompt = "Based on the outline given and contecxxt of reseach write short draft article."
    )
)

In [36]:
reviewer = Agent(
    AgentConfig(
        name = "reviewer",
        model = modelconfig,
        prompt = "Review the draft for correctness and return the imporved final version"
    )
)

In [37]:
def run_research(state:AgentState):
    response = researcher.invoke(f"""
    Research on the following topic
    {state['query']}
    """)
    return {"researcher" : response}

def run_outline(state:AgentState):
    response = outliner.invoke(f"""
    Outline the below topci: 
    {state['researcher']}
    """)
    return {"outliner" : response}

def run_writer(state:AgentState):
    response = researcher.invoke(f"""
    Wirter on below oujtline
    {state['outliner']}
    """)
    return {"writer" : response}

def run_review(state:AgentState):
    response = researcher.invoke(f"""
    Wirter on below oujtline
    {state['writer']}
    """)
    return {"review" : response}

In [38]:
pipeline = WorkflowBuilder(name = "ResearchPipeline")
(
    pipeline
    .start("research").run(run_research)
    .then("outliner").run(run_outline)
    .then("writer").run(run_writer)
    .finish("reviewwer").run(run_review)
)

In [39]:
result = pipeline.execute(state={"query":"Impact of AI in modern logictics"})

In [52]:
pprint(result)

{'messages': [],
 'outliner': [HumanMessage(content="\n    Outline the below topci: \n    [HumanMessage(content='\\n    Research on the following topic\\n    Impact of AI in modern logictics\\n    ', additional_kwargs={}, response_metadata={}, id='832322da-09d0-4e34-b552-7663ed81854a'), AIMessage(content='Here are 3-5 key facts about the impact of AI in modern logic:\\n\\n1. **Automated Theorem Proving**: AI algorithms can now automatically prove mathematical theorems, a task that was previously limited to human mathematicians. This has significant implications for fields like mathematics, philosophy, and computer science.\\n\\n2. **Logical Inference**: AI-powered systems can perform logical inference with unprecedented speed and accuracy, enabling them to draw conclusions from complex sets of premises more efficiently than humans.\\n\\n3. **Resolution-Based Theorem Proving**: AI-based resolution theorem proving is a technique that has been shown to be highly effective in solving probl

In [55]:
pprint(result['review'][-1].content)

('Here are the 3-5 key facts about the impact of AI in modern logic:\n'
 '\n'
 '1. **Automated Theorem Proving**: AI algorithms can now automatically prove '
 'mathematical theorems.\n'
 '2. **Logical Inference**: AI-powered systems can perform logical inference '
 'with unprecedented speed and accuracy.\n'
 '3. **Resolution-Based Theorem Proving**: AI-based resolution theorem proving '
 'is a technique that has been shown to be highly effective in solving '
 'problems in propositional and predicate logic.\n'
 '4. **Knowledge Representation**: AI can now represent complex knowledge '
 'structures using formal languages like ontologies and description logics.\n'
 '\n'
 '(Note: I stopped here as per your request)')
